# VadCLIP + Nhất Quán Theo Dịch Chuyển — Vòng 3

Notebook này làm hai việc mà vòng 2 còn nợ.

**Việc thứ nhất, và nên chạy trước mọi thứ khác: đo độ nhạy dịch chuyển cho năm
checkpoint vòng 2 đã có.** Vòng 2 dừng ở giữa mục 7 nên chưa bao giờ chạy phép đo này.
Toàn bộ giá trị khoa học của mười tiếng GPU đó đang nằm trong năm file `.pth` chưa được
đọc. Phép đo không huấn luyện lại gì cả, chỉ nạp trọng số rồi chấm điểm trên tập test,
nên tốn vài phút mỗi mô hình. Mục 6 tự tìm những checkpoint có mặt trên đĩa và bỏ qua
những cái chưa chạy, nên chạy được ngay dù bộ checkpoint còn thiếu.

**Việc thứ hai: sửa lỗi thiết kế của cách hiệu chuẩn λ.** Vòng 2 giải λ đúng một lần, ở
bước 100, dựa trên mất mát nhiệm vụ của một mô hình gần như chưa được huấn luyện. Sau đó
mất mát nhiệm vụ giảm khoảng năm lần còn mất mát nhất quán giảm mười tới bốn mươi lần,
nên tỉ lệ đóng góp thực tế trôi xa khỏi mục tiêu:

| tỉ lệ đặt ra | epoch 1 | epoch 5 | epoch 10 |
|---|---|---|---|
| 0,01 | 1,02 % | 1,88 % | 1,94 % |
| 0,03 | 2,97 % | 2,42 % | 2,58 % |
| 0,10 | 9,82 % | 4,44 % | 3,99 % |

Ba lần chạy lẽ ra cách nhau mười lần về liều thì đến cuối chỉ còn cách nhau khoảng hai
lần. Đường cong λ của vòng 2 vì thế không phân biệt được gì — không phải vì phương pháp
trơ với λ, mà vì λ chưa bao giờ thật sự thay đổi đủ nhiều.

Cờ mới `--lambda-auto-recalibrate` giải lại λ ở cuối mỗi epoch, giữ tỉ lệ đóng góp đúng
bằng con số đặt ra suốt cả quá trình. Đi kèm là `--lambda-auto-max-growth`: giữ một tỉ lệ
cố định trên một mẫu số đang tiến về 0 là đòi hỏi một λ vô hạn, nên vòng lặp phản hồi này
cần dây cương. Mặc định 2,0, tức mỗi lần hiệu chuẩn λ chỉ được đổi tối đa gấp đôi hoặc
giảm một nửa. Cả hai cờ mặc định tắt, nên mọi lần chạy vòng 2 vẫn tái lập nguyên vẹn.

## Thứ tự chạy

Mục 1 đến 5 là chuẩn bị. **Mục 6 và 7 là phần bắt buộc** — rẻ, và trả lời câu hỏi chính
mà vòng 2 bỏ ngỏ. Mục 8 trở đi là huấn luyện mới, tốn nhiều giờ; chỉ chạy sau khi đã đọc
bảng ở mục 7.

## 1. Mount Drive Và Cấu Hình

Notebook này **đọc lại thư mục `model/` của vòng 2** để lấy checkpoint, và ghi kết quả
mới vào `shift_v3_metrics.csv` cùng `logs_shift_v3`, tách hẳn khỏi số liệu vòng 2.

Bản code trên Drive phải là bản mới có hai cờ hiệu chuẩn lại. Mục 3 kiểm tra điều đó và
dừng hẳn nếu chưa có, thay vì để bạn phát hiện sau ba tiếng huấn luyện.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception:
    print('Not running in Colab or Drive is already mounted.')

PROJECT_ROOT = Path('/content/drive/MyDrive/Finetune VadCLIP')
SRC_DIR = PROJECT_ROOT / 'VadCLIP' / 'src'
LIST_DIR = PROJECT_ROOT / 'VadCLIP' / 'list'
DRIVE_FEATURE_ROOT = PROJECT_ROOT / 'UCFClipFeatures'
DRIVE_FEATURE_ARCHIVES = [
    PROJECT_ROOT / 'UCFClipFeatures.tar',
    PROJECT_ROOT / 'UCFClipFeatures.tar.gz',
    PROJECT_ROOT / 'UCFClipFeatures.tgz',
    PROJECT_ROOT / 'UCFClipFeatures.zip',
]
LOCAL_FEATURE_ROOT = Path('/content/UCFClipFeatures')
FEATURE_ROOT = DRIVE_FEATURE_ROOT

# Chi dung lam moc doi chieu, KHONG dung de khoi tao.
PAPER_MODEL = PROJECT_ROOT / 'model_ucf.pth'

RESULT_DIR = PROJECT_ROOT / 'Result'
LOG_DIR = RESULT_DIR / 'logs_shift_v3'
METRICS_CSV = str(RESULT_DIR / 'shift_v3_metrics.csv')
METRICS_CSV_V2 = str(RESULT_DIR / 'shift_v2_metrics.csv')   # doc de so sanh, khong ghi

TRAIN_LIST = '../list/ucf_CLIP_rgb_relative.csv'
TEST_LIST = '../list/ucf_CLIP_rgbtest_relative.csv'
GT_ARGS = [
    '--gt-path', '../list/gt_ucf.npy',
    '--gt-segment-path', '../list/gt_segment_ucf.npy',
    '--gt-label-path', '../list/gt_label_ucf.npy',
]

# Lich huan luyen giu y nguyen nhu vong 2, de doi chung vong 2 dung lai duoc.
MAX_EPOCH = 10
LR = '2e-5'
USE_PRETRAINED = False
SELECT_METRIC = 'none'
EVAL_STEPS = 0
LAMBDA_AUTO_STEPS = 100
LAMBDA_MAX_GROWTH = 2.0      # moi lan hieu chuan lai, lambda doi toi da gap doi

sys.path.insert(0, str(SRC_DIR))
os.chdir(SRC_DIR)
LOG_DIR.mkdir(parents=True, exist_ok=True)
RESULT_DIR.mkdir(parents=True, exist_ok=True)
Path('model').mkdir(parents=True, exist_ok=True)

PY = [sys.executable, '-u']


def run_command(cmd, log_name=None):
    cmd = [str(part) for part in cmd]
    print('$', ' '.join(cmd), flush=True)
    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                               text=True, bufsize=1)
    captured = []
    for line in process.stdout:
        print(line, end='')
        captured.append(line)
    process.wait()
    output = ''.join(captured)
    if log_name:
        (LOG_DIR / log_name).write_text(output, encoding='utf-8')
        print('Log saved:', LOG_DIR / log_name)
    if process.returncode != 0:
        raise RuntimeError(f'Command failed with exit code {process.returncode}')
    return output


def build_train_cmd(tag, lambda_auto=0.0, lambda_consistency=0.0, seed=234,
                    shift_offset=26, shift_ratio=0.0, shift_direction='head',
                    random_shift=False, ratio_warmup=0, detach=False,
                    branch='c', warmup=1, recalibrate=True,
                    max_growth=None, extra=None):
    return PY + [
        'ucf_train_augment.py',
        '--feature-root', FEATURE_ROOT,
        '--train-list', TRAIN_LIST,
        '--test-list', TEST_LIST,
        *GT_ARGS,
        '--seed', seed,
        '--lambda-consistency', lambda_consistency,
        '--lambda-auto', lambda_auto,
        '--lambda-auto-steps', LAMBDA_AUTO_STEPS,
        '--lambda-auto-recalibrate', str(recalibrate).lower(),
        '--lambda-auto-max-growth',
        LAMBDA_MAX_GROWTH if max_growth is None else max_growth,
        '--shift-offset', shift_offset,
        '--shift-ratio', shift_ratio,
        '--shift-direction', shift_direction,
        '--random-shift', str(random_shift).lower(),
        '--shift-ratio-warmup', ratio_warmup,
        '--consistency-branch', branch,
        '--consistency-detach', str(detach).lower(),
        '--consistency-warmup', warmup,
        '--max-epoch', MAX_EPOCH,
        '--lr', LR,
        '--use-pretrained-model', str(USE_PRETRAINED).lower(),
        '--pretrained-model-path', PAPER_MODEL,
        '--num-workers', 2,
        '--pin-memory', 'true',
        '--eval-steps', EVAL_STEPS,
        '--select-metric', SELECT_METRIC,
        '--run-tag', tag,
        '--metrics-csv', METRICS_CSV,
        '--output-model-path', f'model/model_{tag}.pth',
        '--checkpoint-path', f'model/checkpoint_{tag}.pth',
        '--save-cur-path', f'model/model_cur_{tag}.pth',
        '--epoch-checkpoint-dir', f'model/epoch_checkpoints_{tag}',
    ] + list(extra or [])


def train_shift(tag, **kwargs):
    """Bo qua lan chay da co ket qua: chay lai cell la di tiep, khong phai lam lai."""
    if Path(f'model/model_{tag}.pth').exists():
        print(f'[bo qua] model/model_{tag}.pth da ton tai. Xoa file neu muon chay lai.')
        return None
    return run_command(build_train_cmd(tag, **kwargs), log_name=f'train_{tag}.log')


def shift_sensitivity(tag, model_path=None, offsets=(0, 8, 16, 32)):
    """Chi so CHINH: tuong quan diem so truoc va sau khi dich, sau khi can chinh lai."""
    output_dir = RESULT_DIR / f'shift_sens_{tag}'
    if (output_dir / 'shift_sensitivity_summary.csv').exists():
        print(f'[da co] {output_dir}')
        return output_dir
    run_command(PY + [
        'ucf_shift_sensitivity.py',
        '--feature-root', FEATURE_ROOT,
        '--test-list', TEST_LIST,
        '--model-path', model_path or f'model/model_{tag}.pth',
        '--gt-path', '../list/gt_ucf.npy',
        '--offsets', *offsets,
        '--output-dir', output_dir,
    ], log_name=f'sens_{tag}.log')
    return output_dir


def analyze_per_class(tag, timeline_count=6):
    """Chi so theo lop. Dat va cham -- chi chay cho cau hinh thang cuoc."""
    output_dir = RESULT_DIR / f'perclass_{tag}'
    return run_command(PY + [
        'ucf_analyze_checkpoints.py',
        '--feature-root', FEATURE_ROOT,
        '--test-list', TEST_LIST,
        '--baseline-model-path', PAPER_MODEL,
        '--epoch-checkpoint-dir', f'model/epoch_checkpoints_{tag}',
        '--description-model-path', f'model/model_{tag}.pth',
        '--finetuned-model-type', 'baseline',
        '--output-dir', output_dir,
        *GT_ARGS,
        '--timeline-count', timeline_count,
    ], log_name=f'perclass_{tag}.log')


print('Project root :', PROJECT_ROOT)
print('Source dir   :', SRC_DIR)
print('Metrics CSV  :', METRICS_CSV)
print('Doi chieu    :', METRICS_CSV_V2)

## 2. Dependencies

In [ ]:
!pip -q install ftfy regex tqdm scikit-learn scipy matplotlib pandas
print('done')

## 3. Kiểm Tra File Và Kiểm Tra Phiên Bản Code

Ba việc, và hai việc sau quan trọng hơn việc đầu.

Có đủ file không. Bản code trên Drive có phải bản mới không — lần này danh sách tham số
bắt buộc có thêm `lambda_auto_recalibrate` và `lambda_auto_max_growth`, hai cờ mà mục 8
dựa vào; thiếu chúng thì lệnh huấn luyện chết ngay ở dòng đầu. Và `utils/layers.py` có
phải bản đã vá không — bản gốc ghi cứng `.to('cuda')` trong `DistanceAdj.forward`, nên
trên máy có GPU thì bài kiểm thử chạy mô hình trên CPU sẽ chết vì lệch thiết bị. Kiểm tra
bằng cách chạy thật chứ không đọc mã nguồn.

**Cần upload từ repo cục bộ nếu mục này báo thiếu:**
`VadCLIP/src/ucf_option_augment.py` · `VadCLIP/src/ucf_train_augment.py` ·
`VadCLIP/src/utils/dataset_augment.py` · `VadCLIP/src/utils/layers.py` ·
`VadCLIP/src/ucf_shift_sensitivity.py` · cả thư mục `VadCLIP/src/tests/`.

In [ ]:
import importlib

required_paths = [
    SRC_DIR / 'model.py',
    SRC_DIR / 'ucf_train_augment.py',
    SRC_DIR / 'ucf_option_augment.py',
    SRC_DIR / 'ucf_shift_sensitivity.py',
    SRC_DIR / 'ucf_evaluate.py',
    SRC_DIR / 'ucf_analyze_checkpoints.py',
    SRC_DIR / 'utils' / 'dataset_augment.py',
    SRC_DIR / 'utils' / 'layers.py',
    SRC_DIR / 'utils' / 'tools.py',
    SRC_DIR / 'tests' / 'test_dataset_augment.py',
    SRC_DIR / 'tests' / 'test_shift_consistency_loss.py',
    SRC_DIR / 'tests' / 'test_two_view_batching.py',
    SRC_DIR / 'tests' / 'test_train_smoke.py',
    LIST_DIR / 'ucf_CLIP_rgb_relative.csv',
    LIST_DIR / 'ucf_CLIP_rgbtest_relative.csv',
    LIST_DIR / 'make_gt_ucf_relative.py',
]
gt_paths = [LIST_DIR / n for n in ('gt_ucf.npy', 'gt_segment_ucf.npy', 'gt_label_ucf.npy')]

missing = [str(p) for p in required_paths if not p.exists()]
if missing:
    print('THIEU FILE:')
    for path in missing:
        print('  ', path)
    raise FileNotFoundError('Upload cac file con thieu len Drive roi chay lai cell nay.')

import ucf_option_augment
importlib.reload(ucf_option_augment)
known = {action.dest for action in ucf_option_augment.parser._actions}
needed = {'shift_ratio', 'shift_direction', 'shift_ratio_warmup',
          'lambda_auto', 'lambda_auto_steps', 'select_metric', 'run_tag', 'metrics_csv',
          'lambda_auto_recalibrate', 'lambda_auto_max_growth'}
absent = sorted(needed - known)
if absent:
    print('BAN CODE TREN DRIVE LA BAN CU. Thieu cac tham so:')
    for name in absent:
        print('  --' + name.replace('_', '-'))
    raise RuntimeError(
        'Upload lai ucf_option_augment.py, ucf_train_augment.py, utils/dataset_augment.py '
        'va thu muc tests/ tu repo cuc bo.'
    )

import torch
from utils.layers import DistanceAdj

probe = DistanceAdj()                       # tham so nam tren CPU
probe_out = probe(2, 32)
if probe_out.device.type != 'cpu':
    raise RuntimeError(
        'utils/layers.py tren Drive la BAN CU: DistanceAdj tra ve tensor tren '
        f'{probe_out.device}, dang le phai theo thiet bi cua module (cpu). '
        'Upload lai VadCLIP/src/utils/layers.py tu repo cuc bo.'
    )
del probe, probe_out

GT_MISSING = [str(p) for p in gt_paths if not p.exists()]
print('Du toan bo file bat buoc, va ban code la ban moi (co hai co hieu chuan lai).')
print('utils/layers.py: ban da va.')
print('Feature root ton tai:', DRIVE_FEATURE_ROOT.exists())

existing = sorted(Path('model').glob('model_v2_*.pth'))
print()
print(f'Checkpoint vong 2 tim thay tren Drive: {len(existing)}')
for path in existing:
    print('  ', path.name)
if not existing:
    print('  (khong co) -- muc 6 se khong do duoc gi ngoai checkpoint goc.')

if GT_MISSING:
    print()
    print('Thieu ground truth (muc 4.1 se sinh lai):')
    for path in GT_MISSING:
        print('  ', path)

## 4. Copy Feature Sang Runtime Local — BẮT BUỘC

Feature trên Drive nằm ở dạng file nén. Cell này giải nén nó ra `/content`. Không chạy
cell này thì cả huấn luyện lẫn chấm điểm đều lỗi "missing feature files", vì đọc hàng
nghìn file nhỏ trực tiếp từ Drive vừa chậm vừa hay đứt.

Mục 6 cũng cần feature — nó chấm điểm trên tập test — nên cell này phải chạy kể cả khi
bạn chỉ định làm mục 6 rồi dừng.

In [ ]:
import shutil, time

subprocess.run(['df', '-h', '/content'], check=False)
archive = next((p for p in DRIVE_FEATURE_ARCHIVES if p.exists()), None)

if LOCAL_FEATURE_ROOT.exists() and any(LOCAL_FEATURE_ROOT.rglob('*.npy')):
    print('Da co feature o', LOCAL_FEATURE_ROOT)
elif archive is not None:
    print('Giai nen', archive)
    LOCAL_FEATURE_ROOT.mkdir(parents=True, exist_ok=True)
    start = time.time()
    if archive.suffix == '.zip':
        shutil.unpack_archive(str(archive), str(LOCAL_FEATURE_ROOT))
    else:
        subprocess.run(['tar', '-xf', str(archive), '-C', str(LOCAL_FEATURE_ROOT)], check=True)
    print(f'Xong sau {time.time() - start:.0f}s')
elif DRIVE_FEATURE_ROOT.exists():
    print('Khong co file nen, copy thu muc tu Drive (cham).')
    shutil.copytree(DRIVE_FEATURE_ROOT, LOCAL_FEATURE_ROOT, dirs_exist_ok=True)
else:
    raise FileNotFoundError('Khong tim thay feature o Drive.')

# Bo mot lop thu muc thua neu file nen co san thu muc goc ben trong.
inner = LOCAL_FEATURE_ROOT / 'UCFClipFeatures'
if inner.exists() and not any(LOCAL_FEATURE_ROOT.glob('*/*.npy')):
    for child in inner.iterdir():
        shutil.move(str(child), str(LOCAL_FEATURE_ROOT))
    inner.rmdir()

count = sum(1 for _ in LOCAL_FEATURE_ROOT.rglob('*.npy'))
print('So file .npy:', count)
assert count > 0, 'Giai nen xong nhung khong thay file .npy nao.'
FEATURE_ROOT = LOCAL_FEATURE_ROOT
print('FEATURE_ROOT =', FEATURE_ROOT)

### 4.1. Sinh Lại Ground Truth (chỉ khi mục 3 báo thiếu)

Ba file `gt_*.npy` được sinh từ file nhãn thời gian cộng với độ dài thật của từng file
đặc trưng, nên bắt buộc phải có feature trước.

In [ ]:
if GT_MISSING:
    run_command(PY + [
        str(LIST_DIR / 'make_gt_ucf_relative.py'),
        '--feature-root', FEATURE_ROOT,
        '--test-list', TEST_LIST,
        '--output-dir', LIST_DIR,
    ], log_name='make_gt.log')
else:
    print('Da co du ground truth, bo qua.')

## 5. Unit Test

Bốn file, vài chục giây, chạy trên CPU với một bộ mã hoá CLIP giả nên không cần feature
thật. Bài quan trọng với vòng này nằm trong `test_train_smoke.py`: nó chạy ba epoch với
`--lambda-auto-recalibrate` bật rồi kiểm tra rằng λ **có** đổi giữa các epoch, rằng mỗi
lần đổi nằm trong dây cương, và rằng khi tắt cờ thì λ đứng yên đúng như vòng 2.

In [ ]:
for test_file in ('test_dataset_augment', 'test_shift_consistency_loss',
                  'test_two_view_batching', 'test_train_smoke'):
    run_command(PY + [f'tests/{test_file}.py'], log_name=f'{test_file}.log')

## 6. Đo Độ Nhạy Dịch Chuyển Cho Checkpoint Đã Có — CHẠY TRƯỚC

Đây là phần trả nợ vòng 2, và là phần rẻ nhất trong cả notebook.

Phép đo trả lời đúng câu hỏi mà hàm mất mát nhắm tới: dịch đầu vào đi Δ bước, căn chỉnh
kết quả trở lại trục cũ, thì chuỗi điểm số có giữ nguyên không. Nó **không huấn luyện lại
gì cả** — chỉ nạp trọng số và chấm trên 266 video test.

Vì sao vẫn phải đo, khi log huấn luyện đã cho thấy mất mát nhất quán giảm bốn mươi lần:
con số bốn mươi lần đó đo **trên tập huấn luyện, tại đúng độ dịch Δ = 26 mà mô hình được
luyện**. Nó chứng minh việc tối ưu hoá chạy được, không chứng minh tính bất biến có
chuyển sang video chưa từng thấy và sang những độ dịch chưa từng luyện. Mục này đo ở Δ
bằng 8, 16 và 32 — cả ba đều khác 26.

Cell tự tìm checkpoint trên đĩa thay vì đi theo một danh sách cứng, nên chạy được ngay dù
bộ checkpoint còn thiếu, và chạy lại được sau khi có thêm checkpoint mới mà không đo lại
cái cũ. Một checkpoint hỏng cũng không làm chết cả vòng lặp: nó được ghi vào danh sách
thất bại rồi đi tiếp.

Mốc của vòng 1, để biết trước con số nào là đáng kể: ở Δ = 16, tương quan của đối chứng
là 0,635 còn của checkpoint gốc trong bài báo là 0,697. Khoảng cách 0,062 đó là thang đo
tự nhiên cho mọi chênh lệch ở đây.

In [ ]:
PREFERRED_ORDER = [
    'v2_ctrl_s234', 'v2_ctrl_s1234',
    'v2_lam0.01', 'v2_lam0.03', 'v2_lam0.1', 'v2_lam0.3',
    'v2_cut1_ratio', 'v2_cut2_rand', 'v2_cut3_both', 'v2_cut4_curr',
    'v2_detach', 'v2_best_s1234', 'v2_lam_hi',
]


def discover_tags():
    """Tag co checkpoint tren dia, uu tien thu tu quen thuoc roi den phan con lai.

    Doc tu dia thay vi tu mot danh sach cung: danh sach cung cua vong 2 ke ten muoi ba
    lan chay, ma phien do dut giua chung nen chi nam trong so do ton tai.
    """
    found = {p.stem[len('model_'):] for p in Path('model').glob('model_*.pth')}
    found = {t for t in found if not t.startswith('cur_')}
    ordered = [t for t in PREFERRED_ORDER if t in found]
    rest = sorted(t for t in found if t not in PREFERRED_ORDER)
    return ordered + rest


TAGS = discover_tags()
print(f'Tim thay {len(TAGS)} checkpoint:', ', '.join(TAGS) if TAGS else '(khong co)')
print()

failed = []
try:
    shift_sensitivity('paper', model_path=PAPER_MODEL)
except Exception as error:
    failed.append(('paper', repr(error)))
    print('[LOI] paper:', error)

for tag in TAGS:
    print('=' * 100)
    try:
        shift_sensitivity(tag)
    except Exception as error:
        failed.append((tag, repr(error)))
        print(f'[LOI] {tag}: {error}')

print()
print('=' * 100)
measured = [t for t in ['paper'] + TAGS
            if (RESULT_DIR / f'shift_sens_{t}' / 'shift_sensitivity_summary.csv').exists()]
print(f'Da co bang do nhay cho {len(measured)} mo hinh:', ', '.join(measured))
if failed:
    print()
    print('That bai:')
    for tag, error in failed:
        print(f'  {tag}: {error}')

## 7. Bảng Tổng Hợp

Ghép ba nguồn: bảng độ nhạy vừa đo, dòng cuối của mỗi lần chạy trong CSV huấn luyện vòng
2, và CSV vòng 3 nếu đã có.

CSV vòng 2 có dòng lặp — lần chạy `v2_ctrl_s1234` bị ngắt rồi chạy lại nên epoch 1 xuất
hiện ba lần, epoch 2 hai lần. Cell này bỏ trùng theo cặp (lần chạy, epoch) và giữ dòng
cuối.

Cột quan trọng nhất là `d_corr16`, không phải `d_auc`. Sàn nhiễu đo được ở vòng 2 là
**2,56 điểm AUC** giữa hai lần chạy đối chứng chỉ khác hạt giống — lớn hơn mọi hiệu ứng
mà λ tạo ra. AUC ở đây là ràng buộc không được xấu đi, không phải mục tiêu.

In [ ]:
import pandas as pd

OFFSETS = [8, 16, 32]
NAN = float('nan')


def ctrl_of(tag):
    return 'v2_ctrl_s1234' if tag.endswith('_s1234') else 'v2_ctrl_s234'


def read_sensitivity(tag):
    path = RESULT_DIR / f'shift_sens_{tag}' / 'shift_sensitivity_summary.csv'
    if not path.exists():
        return None
    frame = pd.read_csv(path).set_index('offset')
    row = {f'corr{o}': float(frame.loc[o, 'mean_classifier_corr'])
           for o in OFFSETS if o in frame.index}
    row['auc_spread'] = float(frame.iloc[0]['classifier_auc_spread_common'])
    return row


def read_train_metrics(*csv_paths):
    """Dong cuoi cua moi lan chay.

    Bo trung theo (run, epoch): mot lan chay bi ngat roi chay lai se NOI THEM vao cuoi
    file chu khong ghi de, nen vong 2 co epoch 1 ba lan va epoch 2 hai lan.
    """
    out = {}
    for csv_path in csv_paths:
        if not Path(csv_path).exists():
            continue
        frame = pd.read_csv(csv_path)
        frame = frame[frame.run != 'run']                      # dong tieu de bi lap
        frame = frame.drop_duplicates(subset=['run', 'epoch'], keep='last')
        for tag, group in frame.groupby('run'):
            last = group.sort_values('epoch').iloc[-1]
            out[tag] = {'auc': float(last['auc']) * 100, 'ap': float(last['ap']) * 100,
                        'lambda_used': float(last['lambda_used']),
                        'epochs': int(last['epoch'])}
    return out


def build_table():
    """Bang chinh, ghep do nhay voi chi so huan luyen, kem delta so voi doi chung CUNG SEED."""
    metrics = read_train_metrics(METRICS_CSV_V2, METRICS_CSV)
    rows = []
    for tag in ['paper'] + discover_tags():
        sens = read_sensitivity(tag)
        if sens is None:
            continue
        rows.append({'run': tag, **sens, **metrics.get(tag, {})})
    if not rows:
        return pd.DataFrame()
    frame = pd.DataFrame(rows).set_index('run')
    for tag in frame.index:
        base = ctrl_of(tag)
        if tag == 'paper' or tag.startswith('v2_ctrl') or base not in frame.index:
            continue
        for col in ('corr8', 'corr16', 'corr32', 'auc'):
            if col in frame.columns:
                frame.loc[tag, 'd_' + col] = frame.loc[tag, col] - frame.loc[base, col]
    return frame


def noise_floor(frame):
    """Chenh lech giua hai lan chay doi chung: thuoc do duy nhat de doc moi delta khac."""
    out = {}
    if {'v2_ctrl_s234', 'v2_ctrl_s1234'} <= set(frame.index):
        for col in ('corr8', 'corr16', 'corr32', 'auc', 'auc_spread'):
            if col in frame.columns:
                out[col] = abs(frame.loc['v2_ctrl_s234', col]
                               - frame.loc['v2_ctrl_s1234', col])
    return out


table = build_table()
if not len(table):
    print('Chua co bang do nhay nao. Chay muc 6 truoc.')
else:
    print('=== Bang chinh ===')
    print(table.round(4).to_string())

    print()
    print('=== San nhieu: chenh lech giua hai lan chay doi chung ===')
    noise = noise_floor(table)
    if noise:
        for col, value in noise.items():
            print(f'  {col:<12} {value:.4f}')
        print('  (AUC: vong 2 do duoc 2,56 diem. Moi hieu ung nho hon con so nay')
        print('   deu khong ket luan duoc.)')
    else:
        print('  Chua co du hai doi chung -- khong co thuoc do.')

    print()
    print('=== Khoang cach toi checkpoint goc, lam thang do ===')
    if 'paper' in table.index and 'v2_ctrl_s234' in table.index:
        for col in ('corr8', 'corr16', 'corr32'):
            if col in table.columns:
                gap = table.loc['paper', col] - table.loc['v2_ctrl_s234', col]
                print(f'  {col:<8} paper {table.loc["paper", col]:.4f} - '
                      f'ctrl {table.loc["v2_ctrl_s234", col]:.4f} = {gap:+.4f}')
        print('  Mot cau hinh dong duoc phan lon khoang cach nay la mot ket qua that.')

    short = table[table['epochs'].notna() & (table['epochs'] < MAX_EPOCH)]         if 'epochs' in table.columns else table.iloc[0:0]
    if len(short):
        print()
        print('=== CANH BAO: lan chay co it epoch hon lich huan luyen ===')
        for tag in short.index:
            print(f'  {tag}: CSV chi ghi den epoch {int(short.loc[tag, "epochs"])}/{MAX_EPOCH}, '
                  f'nen cot AUC/AP o tren KHONG phai gia tri cuoi cung.')
        print('  Neu log cho thay lan chay da xong du epoch thi file CSV bi cat khi chep')
        print('  ve, khong phai lan chay bi dut. Chep lai file tu Drive roi chay lai cell.')

### 7.1. Đường Cong λ Của Vòng 2, Đọc Riêng

Ba điểm đã chạy — 0,01 · 0,03 · 0,10 — đều **không** hiệu chuẩn lại, nên liều thực tế của
chúng ở cuối lần lượt chỉ là 1,9 % · 2,6 % · 4,0 %. Hãy đọc bảng này với con số đó trong
đầu: nếu `corr16` gần như không đổi qua ba điểm thì đó là điều phải xảy ra, vì ba điểm
gần nhau hơn nhãn của chúng gợi ý rất nhiều. Kết luận thật nằm ở mục 8.

In [ ]:
if len(table):
    print(f"{'ty le':>8} {'lambda giai duoc':>18} {'corr8':>9} {'corr16':>9} "
          f"{'d_corr16':>10} {'AUC':>8} {'d_auc':>8}")
    for ratio in [0.01, 0.03, 0.10, 0.30]:
        tag = f'v2_lam{ratio:g}'
        if tag not in table.index:
            continue
        r = table.loc[tag]
        print(f"{ratio:>8g} {r.get('lambda_used', NAN):>18.4e} {r.get('corr8', NAN):>9.4f} "
              f"{r.get('corr16', NAN):>9.4f} {r.get('d_corr16', NAN):>+10.4f} "
              f"{r.get('auc', NAN):>8.2f} {r.get('d_auc', NAN):>+8.2f}")
    print()
    print('  Lieu thuc te o cuoi huan luyen, do tu log vong 2: 1,9% / 2,6% / 4,0%')
    print('  Vong 1 chay o ty le 6,5e-06 den 8,7e-05, thap hon ca diem thap nhat o day.')

## 8. Đường Cong λ Vòng 3 — Có Hiệu Chuẩn Lại

Bốn lần chạy, giữ nguyên cách cắt của vòng 1 (Δ = 26 cố định, cắt đầu, không ngẫu nhiên)
để biến duy nhất là liều. Khác vòng 2 ở đúng một điểm: λ được giải lại ở cuối mỗi epoch,
nên tỉ lệ đóng góp giữ đúng con số đặt ra thay vì trôi xuống còn một phần ba.

**Dải chọn rộng hơn hẳn: 1,00 · 0,30 · 0,10 · 0,03.** Vòng 2 đã cho thấy vùng dưới 4 %
không tạo ra khác biệt đo được, nên kéo dài thêm về phía đó là lãng phí GPU. Tỉ lệ 1,00
nghĩa là số hạng nhất quán nặng ngang toàn bộ phần còn lại của mục tiêu — gần như chắc
chắn quá tay, và đó là chủ ý: nó xác định **trần**, tức điểm mà mô hình bắt đầu chọn
nghiệm tầm thường là cho ra điểm số phẳng theo thời gian. Một đường cong có trần đọc được
nhiều hơn một đường cong toàn điểm an toàn.

Thứ tự chạy là từ liều cao xuống thấp, không phải từ thấp lên cao. Lý do thực dụng: nếu
phiên Colab đứt giữa chừng thì thứ còn lại trên đĩa là đầu có tín hiệu mạnh nhất, chứ
không phải bốn tiếng ở vùng đã biết là không có gì. Mỗi lần chạy tự bỏ qua nếu checkpoint
đã tồn tại, nên chạy lại cell là đi tiếp chứ không phải làm lại.

Dây cương đặt ở 2,0. Vòng 2 cho thấy mất mát nhiệm vụ giảm khoảng năm lần trong khi mất
mát nhất quán giảm mười tới bốn mươi lần, nên λ cần lớn lên khoảng tám lần trong mười
epoch — thoải mái nằm trong giới hạn. Dây cương ở đây là lưới an toàn phòng khi mất mát
nhất quán sập về gần 0, không phải một ràng buộc đang siết.

**Chi phí: khoảng bốn lần một tiếng rưỡi.** Nếu chỉ chạy được một phần, hai lần đầu (1,00
và 0,30) là đủ để biết liều lớn có tác dụng hay không.

In [ ]:
LAMBDA_TARGETS_V3 = [1.00, 0.30, 0.10, 0.03]   # cao xuong thap, co chu y

for ratio in LAMBDA_TARGETS_V3:
    print('=' * 100)
    print(f'--- ty le dong gop {ratio:g}, hieu chuan lai moi epoch ---')
    train_shift(f'v3_lam{ratio:g}', lambda_auto=ratio, seed=234, recalibrate=True)

### 8.1. λ Đã Đi Những Đâu

Trước khi đọc chất lượng, hãy đọc quỹ đạo của chính λ. Cell này bóc dòng `[lambda-auto]`
ra khỏi log và cho thấy hai thứ: λ có thật sự lớn lên theo từng epoch không, và tỉ lệ
đóng góp thực tế có bám sát mục tiêu không.

Nếu cột tỉ lệ thực nằm sát mục tiêu suốt mười epoch thì bản sửa đã làm đúng việc của nó.
Nếu nó vẫn trôi xuống, hãy xem λ có bị dây cương chặn không — dấu hiệu là chữ `capped`
trong log — và nếu có thì nới `LAMBDA_MAX_GROWTH` lên 3,0 rồi chạy lại.

In [ ]:
import re

pattern = re.compile(
    r'end of epoch (\d+): realised share ([\d.]+) vs target ([\d.]+) '
    r'\| lambda ([\d.e+-]+) -> ([\d.e+-]+)(.*)'
)
first_solve = re.compile(
    r'step \d+: task [\d.]+ \| consistency [\d.e+-]+ -> lambda = ([\d.e+-]+)'
)

for ratio in LAMBDA_TARGETS_V3:
    tag = f'v3_lam{ratio:g}'
    log_path = LOG_DIR / f'train_{tag}.log'
    if not log_path.exists():
        print(f'[chua chay] {tag}')
        continue
    text = log_path.read_text(encoding='utf-8', errors='replace')
    print(f'--- {tag} (muc tieu {ratio:g}) ---')
    initial = first_solve.search(text)
    if initial:
        print(f'  lambda ban dau: {float(initial.group(1)):.4e}')
    matches = pattern.findall(text)
    if not matches:
        print('  (khong co dong hieu chuan lai -- lan chay nay tat co, hoac la doi chung)')
    else:
        print(f"  {'epoch':>6} {'ty le thuc':>12} {'lambda ->':>14}  ghi chu")
        for epoch, share, target, before, after, note in matches:
            print(f"  {epoch:>6} {float(share):>12.4f} {float(after):>14.4e}  {note.strip()}")
    print()

## 9. Đo Độ Nhạy Cho Các Lần Chạy Mới, Và So Vòng 2 Với Vòng 3

Cell này chạy lại đúng logic tự tìm của mục 6 — checkpoint nào chưa đo thì đo, đã đo rồi
thì bỏ qua — nên gọi được bất cứ lúc nào, kể cả khi mục 8 mới chạy được một nửa.

Bảng cuối đặt hai vòng cạnh nhau ở cùng tỉ lệ đặt ra. Đó là phép so sánh có kiểm soát cho
chính bản sửa: cùng cách cắt, cùng hạt giống, cùng lịch huấn luyện, khác đúng một chuyện
là λ có được giải lại hay không.

In [ ]:
def measure_missing():
    """Do nhung checkpoint chua co bang, bo qua nhung cai da co."""
    for tag in discover_tags():
        if (RESULT_DIR / f'shift_sens_{tag}' / 'shift_sensitivity_summary.csv').exists():
            continue
        print('=' * 100)
        try:
            shift_sensitivity(tag)
        except Exception as error:
            print(f'[LOI] {tag}: {error}')


measure_missing()
table = build_table()
noise = noise_floor(table)

print('=== Vong 2 (hieu chuan mot lan) so voi vong 3 (hieu chuan lai moi epoch) ===')
print(f"{'ty le':>8} {'vong':>6} {'lambda cuoi':>14} {'corr16':>9} {'d_corr16':>10} "
      f"{'AUC':>8} {'d_auc':>8}")
for ratio in [0.01, 0.03, 0.10, 0.30, 1.00]:
    for prefix, label in (('v2_lam', 'v2'), ('v3_lam', 'v3')):
        tag = f'{prefix}{ratio:g}'
        if tag not in table.index:
            continue
        r = table.loc[tag]
        print(f"{ratio:>8g} {label:>6} {r.get('lambda_used', NAN):>14.4e} "
              f"{r.get('corr16', NAN):>9.4f} {r.get('d_corr16', NAN):>+10.4f} "
              f"{r.get('auc', NAN):>8.2f} {r.get('d_auc', NAN):>+8.2f}")

print()
print('=== Quyet dinh ===')
candidates = table[[t.startswith('v3_lam') for t in table.index]] if len(table) else table
if 'd_corr16' in getattr(candidates, 'columns', []):
    candidates = candidates[candidates['d_corr16'].notna()]
if len(candidates) and 'corr16' in noise:
    floor = noise['corr16']
    best = candidates['d_corr16'].idxmax()
    gain = float(candidates.loc[best, 'd_corr16'])
    cost = float(candidates.loc[best, 'd_auc'])
    print(f'  Manh nhat : {best}')
    print(f'  d_corr16  : {gain:+.4f}  (san nhieu {floor:.4f}, tuc '
          f'{gain / floor if floor else float("inf"):.1f} lan)')
    print(f'  d_auc     : {cost:+.2f} diem  (san nhieu AUC 2,56 diem)')
    if gain > 3 * floor and cost > -0.5:
        print('  => Phuong phap co tac dung that, khong danh doi gi dang ke.')
    elif gain > 3 * floor:
        print('  => Co tac dung nhung DANH DOI voi AUC. Bao cao la mot danh doi.')
    else:
        print('  => Khong vuot duoc san nhieu. Ket luan am tinh -- van co gia tri, xem muc 11.')
    print()
    print('  BEST_RATIO cho muc 10 =', best.replace('v3_lam', ''))
else:
    print('  Chua du du lieu. Chay xong muc 8 va cell tren truoc.')

## 10. Thang Cắt — Cộng Dồn Từng Cải Tiến Một

Chỉ chạy sau khi mục 9 đã chốt một tỉ lệ. Bốn lần chạy ở tỉ lệ đó, mỗi lần thêm đúng một
thay đổi vào cách cắt, để biết cải tiến nào đóng góp bao nhiêu.

Δ theo tỉ lệ độ dài từng video thay vì 26 bước cố định. Δ = 26 là 10 % của lưới nhưng
khoảng 19 % của video trung vị, và có 3,9 % số video ngắn tới mức bị cắt sạch, không đóng
góp gì cho hàm mất mát — con số 3,9 % đó đo được trong log vòng 2 và ổn định qua mọi
epoch. Cắt theo tỉ lệ làm liều đồng đều và xoá luôn phần bị mất, miễn phí.

Rồi độ lớn ngẫu nhiên, để ràng buộc bất biến ở mọi khoảng cách chứ không chỉ một.

Rồi dịch hai chiều, để không phải lúc nào đầu video cũng là phần bị cắt.

Rồi curriculum trên độ lớn: dịch nhẹ trước, mạnh dần.

**Đặt `BEST_RATIO` bằng con số mục 9 in ra trước khi chạy cell.**

In [ ]:
# Dat gia tri muc 9 in ra vao day truoc khi chay.
BEST_RATIO = 0.30

common = dict(lambda_auto=BEST_RATIO, seed=234, recalibrate=True)

train_shift('v3_cut1_ratio', shift_ratio=0.1, shift_offset=0, **common)
train_shift('v3_cut2_rand', shift_ratio=0.1, shift_offset=0, random_shift=True, **common)
train_shift('v3_cut3_both', shift_ratio=0.1, shift_offset=0, random_shift=True,
            shift_direction='both', **common)
train_shift('v3_cut4_curr', shift_ratio=0.1, shift_offset=0, random_shift=True,
            shift_direction='both', ratio_warmup=4, **common)

### 10.1. Thang Cắt, Đọc Riêng

Bằng chứng ở đây nằm ở **hình dạng**, không ở từng giá trị. Một thang đi lên đều qua bốn
bậc có sức nặng hơn bất kỳ chênh lệch đơn lẻ nào, vì nhiễu ngẫu nhiên không tạo ra hình
dạng đó. Ngược lại, nếu bốn bậc nhảy loạn xạ quanh nhau thì kết luận đúng là "không phân
biệt được", bất kể bậc nào tình cờ cao nhất.

In [ ]:
measure_missing()
table = build_table()

ladder = [(f'v3_lam{BEST_RATIO:g}', 'goc: offset 26 co dinh'),
          ('v3_cut1_ratio', '+ ty le theo do dai'),
          ('v3_cut2_rand', '+ do lon ngau nhien'),
          ('v3_cut3_both', '+ dich hai chieu'),
          ('v3_cut4_curr', '+ curriculum')]

print(f"{'cau hinh':>16} {'them gi':<26} {'corr16':>9} {'d_corr16':>10} "
      f"{'AUC':>8} {'d_auc':>8}")
for tag, what in ladder:
    if tag not in table.index:
        print(f"{tag:>16} {what:<26} {'(chua chay)':>9}")
        continue
    r = table.loc[tag]
    print(f"{tag:>16} {what:<26} {r.get('corr16', NAN):>9.4f} "
          f"{r.get('d_corr16', NAN):>+10.4f} {r.get('auc', NAN):>8.2f} "
          f"{r.get('d_auc', NAN):>+8.2f}")

## 11. Cách Đọc Kết Quả

**Chỉ số chính là `d_corr16`, không phải `d_auc`.** Vòng 2 đo được sàn nhiễu AUC là 2,56
điểm giữa hai lần chạy chỉ khác hạt giống, trong khi hiệu ứng lớn nhất mà λ tạo ra là
0,57 điểm. AUC ở đây là ràng buộc không được xấu đi, không phải mục tiêu. Một kết luận
dựa trên chênh lệch AUC dưới sàn nhiễu thì không phải kết luận.

Ba mức nhiễu đo được ở vòng 2, để đối chiếu: trong cùng một lần chạy, AUC dao động 0,07
điểm giữa các epoch cuối; cùng cấu hình cùng hạt giống nhưng khác phần cứng thì lệch 0,23
điểm; khác hạt giống thì lệch 2,56 điểm.

**Cảnh giác với nghiệm tầm thường ở tỉ lệ 1,00.** Nếu `corr16` vọt lên gần 1 trong khi
AUC tụt mạnh thì đó không phải thành công: mô hình đã học cách cho ra điểm số gần như
hằng số theo thời gian, thoả mãn hàm nhất quán một cách hoàn hảo và phá hỏng khả năng
định vị. Cột `auc_spread` và AUC tuyệt đối là thứ phát hiện ra điều này. Đó chính là trần
mà tỉ lệ 1,00 được đưa vào để đi tìm.

**Nếu λ vẫn không tạo ra khác biệt sau khi đã hiệu chuẩn lại, đó là kết quả, không phải
thất bại.** Nó nói rằng bài toán này không khai thác trục thời gian nhiều như trực giác
gợi ý. Đã có công bố trên chính UCF-Crime báo cáo rằng xáo trộn thứ tự thời gian của các
vector đặc trưng không làm giảm độ chính xác; một kết quả âm tính đo cẩn thận ở đây, với
giao thức sạch và một dải liều rộng hai bậc độ lớn, là bằng chứng độc lập cho cùng điều
đó. Kèm hai con số của vòng 1 và vòng 2 — rằng vòng 1 chạy ở tỉ lệ 1e-5, và rằng vòng 2
tưởng chạy dải mười lần nhưng thực tế chỉ hai lần — thì câu chuyện đầy đủ và tự nhất
quán: phương pháp đã được thử ba lần, mỗi lần với một liều lớn hơn hẳn, và lần nào cũng
biết chính xác liều là bao nhiêu.

**Hai quan sát từ vòng 2 nên đưa vào báo cáo dù kết quả cuối ra sao.** Thứ nhất, ở lần
chạy đối chứng, mất mát nhất quán **tăng** đều theo thời gian huấn luyện, từ 0,0037 lên
0,0121 qua mười epoch: huấn luyện thông thường càng lâu thì mô hình càng giòn với dịch
chuyển thời gian. Đó là động cơ của cả hướng nghiên cứu này, và giờ nó là số liệu đo được
chứ không phải suy đoán. Thứ hai, luật chọn checkpoint đổi được dấu của kết luận: lần
chạy λ 0,01 đạt đỉnh 87,97 ở epoch 5, cao hơn đỉnh 87,84 của đối chứng, nhưng kết thúc ở
87,69, thấp hơn 87,84. Cùng một cặp lần chạy, chọn theo AUC tốt nhất thì "thắng" 0,13
điểm, giữ trọng số cuối thì "thua" 0,15 điểm.

## Việc còn lại sau vòng này

Hạt giống thứ ba, để biến sàn nhiễu từ khoảng cách giữa hai điểm thành một ước lượng có
biên độ. Với sàn 2,56 điểm đo trên đúng hai mẫu, con số đó hiện mới là một quan sát chứ
chưa phải một thống kê.

Quét λ lại quanh đỉnh **trên cách cắt mới**, vì đổi cách cắt là đổi thang của mất mát
nhất quán, và có thể dời đỉnh đi.

Ràng buộc nhánh A bằng `--consistency-branch both`, để xem tính nhất quán có chuyển sang
mAP hay không. mAP là chỗ đáng nghi nhất hiện giờ: nó giảm đơn điệu theo λ ở vòng 2, từ
8,15 xuống 7,03, tức khoảng mười ba phần trăm. Con số đó vẫn nằm dưới sàn nhiễu của chính
nó — 1,58 điểm giữa hai đối chứng — nhưng nó là xu hướng đơn điệu duy nhất đi ngược chiều
mong muốn, và nó nằm ở đúng chỉ số mà hàm mất mát này lẽ ra phải giúp.